# Taller en clase · Calidad de datos: diagnosticar, limpiar y medir
**Ciencia de Datos · Ingeniería Industrial · CORHUILA 2026‑B · Semana 9**

El 26 de septiembre diste tus primeros pasos: cargaste un CSV, lo graficaste y le hiciste una limpieza mínima. Hoy vas un paso más allá. Vas a **medir** la calidad de un registro de la planta, **corregir cada problema con la técnica adecuada**, **dejar constancia** de lo que hiciste y **comprobar con números** que los datos quedaron mejor.

Al terminar sabrás:

1. Diagnosticar la calidad de una tabla con cinco dimensiones: **completitud, unicidad, consistencia, formato y validez**.
2. Quitar **duplicados** y **homologar** textos escritos de varias formas.
3. Convertir **tipos de datos**: números guardados como texto y fechas en dos formatos.
4. Normalizar **unidades** (°F → °C).
5. Decidir entre **eliminar o imputar** un dato faltante.
6. Distinguir un dato **atípico** de un dato **imposible**.
7. Medir la calidad **antes y después** y llevar una **bitácora de limpieza**.
8. Responder la pregunta del negocio con datos confiables.

> **Todo el código ya está escrito y funciona de principio a fin.** Tu trabajo es ejecutar cada celda en orden (**Shift + Enter**), leer el resultado y responder las preguntas. Si algo falla, casi siempre es porque se saltó una celda: usa **Ejecutar todo** (*Run All*) desde el inicio. Si ejecutaste una celda de limpieza dos veces y tus cifras ya no coinciden con los puntos de control, usa **Entorno de ejecución → Reiniciar y ejecutar todo**.

| Símbolo | Qué significa |
|---|---|
| 👉 | La línea clave del paso: la que hace el trabajo importante |
| 🧠 | Pregunta para ti: haz doble clic en la celda de texto que sigue y escribe tu respuesta |
| ✅ | Punto de control: compara tu resultado con el número indicado |

## Parte 0 · Preparar el entorno
Dos celdas dejan todo listo:

1. La primera revisa que estén las librerías **pandas** (tablas) y **matplotlib** (gráficos). En Colab ya vienen; en VS Code las instala sola la primera vez si faltan (tarda un minuto).
2. La segunda **crea el archivo de datos** del taller, `registro_produccion.csv`. Los datos viajan dentro del cuaderno, comprimidos, para que el taller funcione aunque no haya internet. **No tienes que entender esa celda**: solo ejecútala.

In [ ]:
import importlib.util, subprocess, sys

faltan = [lib for lib in ["pandas", "matplotlib"] if importlib.util.find_spec(lib) is None]
if faltan:                                   # solo pasa la primera vez en VS Code
    print("Instalando:", faltan)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])

import pandas as pd                 # tablas de datos
import matplotlib.pyplot as plt     # gráficos

plt.rcParams["figure.figsize"] = (9, 4)     # tamaño de los gráficos
pd.set_option("display.max_columns", 20)    # mostrar todas las columnas
print("pandas", pd.__version__, "· listo para trabajar")

In [ ]:
# Crea el archivo registro_produccion.csv a partir de los datos comprimidos (no hace falta entenderla)
import base64, gzip

DATOS = (
    "H4sIAAAAAAACCpVaTY5cNw7e9ykKtVa3RVK/S2MwwGyS2cwFGnYHycJ24rQPMkeZM+RiQ1KqalJPD+gggBGjPkuUSH7kR71f"
    "Xj79+hxef3z/+i18ef7jx29fn8Pv3799/vHpt8/Pf4bPL7+8fHr98e1P/v/Xly+/v3x/ZuzzA0Ysj7E/xhp+ev7rf8/8r356"
    "jBAAC4RA+QlOIBggthyghBMAXWSRGHoP1J/iCSrxMpn/aIHgCR9i/RD7B4GG/zx///wyjYkFA0YxJ9l17hA2hpfh/QKlp7yF"
    "EP/aW4AK4ZoopOsWxZZAK7LXlSgUB/r526dfbwZh7AH4YOTv5w5Rg3pACj0yxBxrQL6oQRBjSHzy8tS2q4g1qHdM8am8Qdri"
    "KyhNTO7JrtMWb1GNAZuYTCcgsYiviG+Imj1XWxyW+FhAYlO3oFfjMIQaEOVkzmzjsF6qLpK9zQMxr6fLBfAVxR1m3g8bbEOw"
    "OTcBMKTKmdMWgmIpqSfBn/gOIYm+HEhXKVsI70HYNYYjx/Ad0teUqkV9idaavriJd5ugcgISixrvlQV0tp3kFaBmBNqTdZ9X"
    "mSFJILSFyAKZ1JP4VLcQtia1HvIaxt27SbytG2ULcSnFjsAUsPszOV/lqL6K/m6sr1JLIWm+1C0khcZcIru8/c4h5h2Vge82"
    "icPzCUgclbsGMDC3Qbzn95JPCTh8PSv5lcRRvJc4KhpHMcg6CgsziRjUtwgxB0Bv2Ka3hYgxHKGZ76by2Y3F1k+xtpkNbiPr"
    "pxI6n9yFjEWImyqzdRk3s4MoFReFxJNVxBQmNT4QNmsKLJ6KHHoIYk1/ANg5QepU6ZpSYH0Oi6cwC10tPofFUxwZozaALSAM"
    "G3d4GUaRkJ/yRT1ipkmEM8vjFiI5zhtx+WQqpgPkcqMd4bfuGYUxPq2ipFWrfhmfVuOmuzfG+Yt34SMdIP/+x7/+eY9iXgUE"
    "Yi4Q10o1/HUCwNC4WxichScYMYYrVKrshRLy9QSWQq95kgVHO67RrlfDpC0uLzbv0PtJODSKK9sRcplpxVUVeMNkacmuw7Zk"
    "UK4ge6x33DD6alZpxnGzJ3JNR1JXMqT8HSq2G/EemW8loZi7t0VXiWKL93Zavc0hPLKTHiCdZKc0msMePFlJWwHhLb5Ha1Fa"
    "s1MYI8vhnU2+keyhCR3jFiE3yAa3hbCTp1H2N+/Eidm2EHZ3F4cqTfSrxVhvxjbKs1CtuRzfQoLW3ujNta4CXsU3CxYgFCGh"
    "x+1htTSa186kxVnk4wlIbOVCJs0UWCbJLDa+8n+XaQ+VpC2t7bj8SlLrUtJ2HuwNZucoSBpflL1FzlOEA+N4NtsWsjOz8fm5"
    "4qXtKmkUxSXS80KgRfp4it7YAbnMq+lNGT/6S3aOqhIyAslbiDaRWViCc8rsVBZXEeuckVP1BCRXU7UCtSa9STnpTVCCuEio"
    "n22n14Nh6Y5h0Wb8dwmLZK+w+B6SyQRpCYo3iCohtraIUCzbRSRoUANZecKcybmq1+mGbFdxKSW1OR7u2DGxeGppii1ENaJ2"
    "bLyKuZZVQXPq6akTnwnqCftJ/mJaCsx9pcu9OORQ45Bl+6XEHlaTNx0Nex0NU2pnb7ereEwFQ0f3LUT7yCJNQ0jA/tztlGaD"
    "AwszeQ0tl9iZJKMjSS+iY67zTHkLoVEVl3TwElr8mUNL1pJ2mHbE2T3XE5Aa0yYB5hOQuIo7E7ngs81U0xe9GwlTaDtH9Tpq"
    "okvNdsipAUlbiJgynOX1M3j9TKA18yruulqQK1V9pPiyjnUUcVHkYN/+rGVBhgrc3vl7cf1+41xZhNmqoDWqaOnku8sW8VNN"
    "2ge4jmKR0J2D71EktLu/fuj3b6MpsiCnzPhYiMNV0L0zbwYRtWO98xoaOLa4N+FKtYdwB98EpnOpeLUYx4C1h4O5jv+ERfvB"
    "FMd/Rdtr5ZEdRHzVddzh+A8XbabFjNdzFOBBGsZd+c9SNh60mSDAV2kDutzKJx56XwTvLA4LXJIGvTLDDgpBc0HolRmyspD2"
    "OJtrthAdkeXZ3KGFWFdRbFMlwxYicSOtOvphBi6yIY1iFU82Ev4TzbUUKzwKs9FqWoWCeGjV0zS5noBojEZJ4jRbdYa4Dj24"
    "8GV/tHcwHKKvZ3xBWHz1RK/gJLF0nGNjB70/Wfpz3nB1xS0kXcTcEtoSFXhIPWnMwF+z86cEe/SDEfTSC7m9I+0295A0Z8/Z"
    "tzBMvc6ffTK2nQZ5jPBkurFb3IHmEDYrcztBpKCvd182brugHNZxiZeLXk7yKXMapnQn0surDQnp3ua4Z7eTTpZ6eOSw0CnX"
    "DiOxx8DVmeSdSSqItj9rhak6iWRj+xZDs55FP8+0EGEJwDky4Qjf6mMY6+CtkzoT0TIGH/MkY88qoqWZkpGH7SpwEdEyqfA6"
    "EL2CTpwIUkAc+VkJLcmSaY476hajYcX5lKTh3y+jfVvyPvD6WV4HJJns7MtCJFxIdUdzzJeWQaQYsVCaV9BQceabyYB8aPaL"
    "GCvVma4WZuSxWNSaFEY3k/dryd0U1PeGtGz48a//fvz54/12irKjqw1eQ1OP85FqD9FHoRvtpy2Ehv5lrmZ23EPSEFVMJN02"
    "Fbio6FJmnY9/i6mzJ1CZU9CSdk5ndxktdD/uxLzogTzHneZ6jzI7Dn+C86fPuwvzLEzq4+02QlssknqIXk4etVvR2cpqk/Nm"
    "GevkEwiOYB4pQVuIxruYotVZzlVOxFs6THqc1OYcT7nNCX7cYsQcbJPR8xZCY8zD6XeVh8nrFqTFl0gzUJ8C2glPYs7z8IZv"
    "VwkHMKYn0QbzQcJpRvjHXVxFHLHlsp07fzN9PxfhNF6trfJ8D4m1Y0O6vBq+L37a4T0h6euPM/itvupcI0995ZaxFqeS9RkZ"
    "/DJOYxSaxbFuIUr/VVpo9ugekubDbF0ueBWEMjITPqVTkD62jGbIke76pJp4ewnGYmO+M327a8bRf5BfydVHWaxpgpnPB3B5"
    "VJVGMAthli1EB7tRfCbinbYYNSfP1wm3jHVXkwBatDAur6pDrI1X1X4crE0BlmBRB335UCGFNt4wKZ61MvH2XPV2JDq8q0pr"
    "lb108qBbm7zMJWl9V6XRBI/26g10cR8qjC7NzY8oLhMxmjU7byE00o+ab4osJA2ma16ckn9Yzdoy6UDMcCG5t9WmUeFfiWh9"
    "Wq1yNVh5J+MJl1q5h2I6U4hiwOKr0uZw/Qwk6ox0etnvOXoAzUiWMlpOV5pDOrBDhwFyBTCPbyzKCQTHGBnkCplVr1uQdAic"
    "ePpRSd0izIAOPcSp91amFO5byKTB8XCQtxAaH1ew+7hbgS1Ev3+h+ShgbFnVe0pa1SmegnA88pgZ3gFyG32T+cDiABKFIC8C"
    "qz3OUaCv5G8suUD0SZ4mJceHiB8g+ibkFjWqM/b7iHySl0PQHi1fLcg3nl0/9yhbAI7Jh8yymt/IvYfKR0ggKdWstb5a6Zn9"
    "KnTIqTqrfX6IdF9n/QCozhblbCWpVsJ/aDq0A0g/5tA6LISSrhbmnm+6kZcbCI45MoKbES8gGopZ2SuHugdpfyFP5fIu1S3E"
    "JVZW1fdGlAvkJhT6mBSYa/QzTNG54g7crqIkWGd3Gt/nsXfEz/8BPDhx+IUpAAA="
)
with open("registro_produccion.csv", "wb") as archivo:
    archivo.write(gzip.decompress(base64.b64decode(DATOS)))

# Un CSV es texto plano: así se ven sus primeras líneas por dentro
with open("registro_produccion.csv", encoding="utf-8") as archivo:
    for _ in range(4):
        print(archivo.readline().rstrip())

✅ **Punto de control:** debes ver la línea de encabezados `fecha,turno,maquina,producidas,defectuosas,temperatura` y tres filas de datos. Cada coma separa una columna; por eso se llama *Comma‑Separated Values*.

## Parte 1 · El caso
La misma planta de empaques del Huila que ya conoces tiene **4 máquinas** en dos líneas de producción y trabaja **3 turnos**. Al final de cada turno, el supervisor digita en una hoja de cálculo compartida cuántas unidades produjo cada máquina, cuántas salieron defectuosas y a qué temperatura cerró la máquina. Hay tres supervisores, cada uno con sus costumbres, y nadie revisa lo que se digita.

| Máquina | Nombre | Línea | Instalada en |
|---|---|---|---|
| M‑01 | Extrusora A | Línea 1 · Película | 2019 |
| M‑02 | Extrusora B | Línea 1 · Película | 2021 |
| M‑03 | Selladora C | Línea 2 · Bolsas | 2012 |
| M‑04 | Cortadora D | Línea 2 · Bolsas | 2023 |

El registro cubre **4 semanas**, del lunes 7 de septiembre al sábado 3 de octubre, de lunes a sábado. La gerencia pregunta:

> **¿Qué máquina y qué turno tienen la mayor tasa de defectos? ¿A cuál le hacemos mantenimiento primero?**

### Diccionario de datos
| Columna | Qué significa | Tipo esperado |
|---|---|---|
| `fecha` | Día del registro, formato `aaaa-mm-dd` | Fecha |
| `turno` | Mañana, Tarde o Noche | Texto (3 valores) |
| `maquina` | Código de la máquina: M‑01 a M‑04 | Texto (4 valores) |
| `producidas` | Unidades fabricadas en el turno | Número entero |
| `defectuosas` | Unidades que no pasaron control de calidad | Número entero |
| `temperatura` | Temperatura de la máquina al cerrar el turno, en °C | Número decimal |

### Reglas del negocio (con ellas vamos a validar)
- La capacidad máxima de una máquina es de **1.800 unidades por turno**.
- Las defectuosas van de **0** hasta el número de unidades producidas.
- El sensor de temperatura mide entre **15 y 60 °C**.
- Solo existen **4 máquinas** y **3 turnos**.

🧠 **Responde:** antes de mirar los datos, ¿qué errores esperas encontrar en una hoja que digitan a mano tres personas distintas durante cuatro semanas? Escribe al menos tres.

**Tu respuesta:**
* Fechas en formatos diferentes.
* Nombres de máquinas o turnos escritos de distintas formas.
* Datos faltantes.
* Registros duplicados.
* Números o temperaturas escritos incorrectamente.


## Parte 2 · Cargar y dar una primera mirada
`pd.read_csv()` lee el archivo y lo convierte en un **DataFrame**, la tabla de pandas. Lo guardamos en la variable `df` (por *data frame*).

**Regla de oro de hoy:** `df` es el **original** y no lo vamos a modificar. Así siempre podremos comparar el antes y el después.

In [ ]:
df = pd.read_csv("registro_produccion.csv")     # 👉 carga el archivo en una tabla
print("Filas y columnas:", df.shape)            # (filas, columnas)

✅ **Punto de control:** `(297, 6)`. Son 297 registros y 6 columnas. Ojo: 4 semanas × 6 días × 3 turnos × 4 máquinas daría 288 registros. ¿De dónde salen los demás? Lo descubriremos.

In [ ]:
df.head(10)      # las primeras 10 filas

Las primeras filas casi siempre se ven bien. Una **muestra al azar** suele mostrar más variedad, y más problemas:

In [ ]:
df.sample(12, random_state=7)     # 12 filas al azar (random_state fija el azar para que todos veamos las mismas)

🧠 **Responde:** en la muestra al azar, ¿ves algo raro? Fíjate en cómo están escritas las fechas, las máquinas, los turnos y las temperaturas.

**Tu respuesta:**
Sí. Se observan **fechas en formatos diferentes, máquinas y turnos escritos de distintas formas, y temperaturas con formatos o unidades diferentes**.


### `info()`: el tipo de cada columna
`info()` muestra, por columna, cuántos valores **no vacíos** tiene y de qué **tipo** es:

| Tipo que muestra pandas | Qué significa |
|---|---|
| `int64` | Números enteros |
| `float64` | Números con decimales (o enteros con algún vacío) |
| `object` o `str` | **Texto** (según tu versión de pandas verás uno u otro) |

In [ ]:
df.info()

Lee el resultado con calma:

- Hay **297 entradas** (filas), pero `defectuosas` y `temperatura` tienen **menos** valores no vacíos: **faltan datos**.
- `defectuosas` aparece como `float64` aunque las unidades son enteras. Cuando una columna de números tiene vacíos, pandas la guarda como decimal, porque el vacío (`NaN`, *Not a Number*) es un valor decimal especial.
- `temperatura` aparece como **texto** (`object` o `str`) aunque debería ser un número. Basta con **un solo** valor que pandas no reconozca como número para que trate toda la columna como texto.

In [ ]:
df.describe()    # estadísticas de las columnas numéricas: conteo, promedio, mínimo, máximo...

`describe()` solo resume columnas **numéricas**. Por eso `temperatura` ni siquiera aparece: pandas la considera texto.

🧠 **Responde:** mira los valores `min` y `max` de `producidas` y `defectuosas`. ¿Cuáles violan las reglas del negocio de la Parte 1? ¿Por qué?

**Tu respuesta:**
* `producidas`: viola la regla si el mínimo es negativo o el máximo supera 1.800 unidades por máquina y turno.

* `defectuosas`: viola la regla si el mínimo es negativo o si el máximo supera las unidades producidas.

La producción no puede ser negativa ni superar la capacidad establecida, y las unidades defectuosas deben estar entre 0 y el total producido.


## Parte 3 · Diagnóstico de calidad (antes de tocar nada)
Un buen analista **primero mide y después limpia**. Si no mides el antes, no puedes demostrar que mejoraste. Vamos a revisar la tabla con cinco **dimensiones de calidad**:

| Dimensión | Pregunta que responde | Cómo la medimos hoy |
|---|---|---|
| **Completitud** | ¿Están todos los datos? | Filas sin ningún vacío |
| **Unicidad** | ¿Cada hecho aparece una sola vez? | Filas que no son copia de otra |
| **Consistencia** | ¿Lo mismo se escribe igual siempre? | Máquinas y turnos escritos como en el catálogo |
| **Formato** | ¿Cada dato tiene la forma acordada? | Fechas en formato `aaaa-mm-dd` |
| **Validez** | ¿Los valores son posibles según el negocio? | Filas que cumplen las reglas de la Parte 1 |

Existen otras dimensiones, como la **exactitud** (¿el dato coincide con la realidad?) y la **oportunidad** (¿llegó a tiempo?), pero esas necesitan comparar con otra fuente o con el reloj. Hoy medimos las cinco que se pueden revisar con la tabla sola.

### 3.1 Completitud: ¿cuántos datos faltan?

In [ ]:
vacios = df.isnull().sum()              # 👉 isnull() marca True donde falta un dato; sum() cuenta los True
porcentaje = (vacios / len(df) * 100).round(1)
pd.DataFrame({"vacíos": vacios, "% del total": porcentaje})

✅ **Punto de control:** `defectuosas` tiene **10** vacíos y `temperatura` **6**. Las demás columnas están completas.

### 3.2 Unicidad: ¿hay filas repetidas?
`duplicated()` marca como `True` cada fila que es **copia exacta** de una anterior. Con `keep=False` marca también la original, para ver las parejas juntas.

In [ ]:
print("Filas duplicadas:", df.duplicated().sum())             # 👉

parejas = df[df.duplicated(keep=False)].sort_values(["fecha", "turno", "maquina"])
parejas.head(8)       # cada registro aparece dos veces

✅ **Punto de control:** **9** filas duplicadas. Ya sabes de dónde salen los registros que sobraban: 288 + 9 = 297.

### 3.3 Consistencia: ¿los textos se escriben siempre igual?
`value_counts()` cuenta cuántas veces aparece cada valor distinto de una columna. Si la planta tiene 4 máquinas, deberíamos ver 4 valores.

In [ ]:
print("Valores distintos en 'maquina':", df["maquina"].nunique())     # nunique() = número de valores distintos
df["maquina"].value_counts()

In [ ]:
df["maquina"].value_counts().plot(kind="bar", color="#7f8c8d")
plt.title("¿Cuántas 'máquinas' hay en el registro?")
plt.ylabel("Registros")
plt.show()

✅ **Punto de control:** aparecen **16** "máquinas" distintas cuando solo existen 4. Para una persona, `M-03`, `m-03` y `M3` son la misma máquina; para la computadora son **textos distintos**. Hay incluso valores que se ven iguales pero tienen un espacio escondido al inicio o al final.

In [ ]:
print("Valores distintos en 'turno':", df["turno"].nunique())
df["turno"].value_counts()

✅ **Punto de control:** **12** formas de escribir 3 turnos.

### 3.4 Formato: ¿las fechas siguen el formato acordado?
El formato acordado es `aaaa-mm-dd` (por ejemplo, `2026-09-07`). Buscamos las fechas que traen una barra `/`, señal de que alguien escribió `dd/mm/aaaa`.

In [ ]:
con_barra = df["fecha"].str.contains("/")            # 👉 True si la fecha tiene una "/"
print("Fechas en formato dd/mm/aaaa:", con_barra.sum())
df.loc[con_barra, "fecha"].head(6)

✅ **Punto de control:** **30** fechas escritas como `dd/mm/aaaa`.

### 3.5 Validez: ¿los valores son posibles?
Primero averiguamos **por qué** `temperatura` quedó como texto. `pd.to_numeric(..., errors="coerce")` intenta convertir cada valor a número; lo que no puede convertir lo deja vacío. Los valores que **no estaban vacíos** pero **no se pudieron convertir** son los culpables.

In [ ]:
temp_como_numero = pd.to_numeric(df["temperatura"], errors="coerce")
no_es_numero = temp_como_numero.isna() & df["temperatura"].notna()     # 👉 tenía algo escrito, pero no es un número
print("Temperaturas que no son número:", no_es_numero.sum())
df.loc[no_es_numero, "temperatura"].head(8).tolist()

✅ **Punto de control:** **20** temperaturas escritas con **coma decimal** (`36,5`). En Colombia usamos la coma para los decimales, pero Python usa el **punto**. Para Python, `36,5` no es un número.

Ahora revisamos las **reglas del negocio**. Cada regla es una condición que da `True` o `False` por fila; `sum()` cuenta cuántas filas la violan.

In [ ]:
producidas = df["producidas"]
defectuosas = df["defectuosas"]

print("Producidas por encima de la capacidad (> 1.800):", (producidas > 1800).sum())
print("Defectuosas mayores que las producidas:         ", (defectuosas > producidas).sum())
print("Defectuosas negativas:                           ", (defectuosas < 0).sum())
print("Temperaturas por encima de 60 °C:                ", (temp_como_numero > 60).sum())

✅ **Punto de control:** 2 por encima de la capacidad, 3 con más defectuosas que producidas, 2 negativas y **14** temperaturas por encima de 60 °C. Ninguna máquina de la planta trabaja a 95 °C: sospechamos que esas lecturas están en **grados Fahrenheit**.

### 3.6 El reporte de calidad
Juntamos todo en un **indicador por dimensión**, de 0 a 100 %. Lo escribimos como una **función**: un bloque de código con nombre que podemos volver a usar. La usaremos ahora (antes) y al final (después), y así la comparación será justa porque mide exactamente lo mismo.

`def` define la función; lo que va entre paréntesis (`tabla`) es el dato que recibe, y `return` es lo que entrega.

In [ ]:
MAQUINAS_VALIDAS = ["M-01", "M-02", "M-03", "M-04"]
TURNOS_VALIDOS = ["Mañana", "Tarde", "Noche"]


def reporte_calidad(tabla):
    """Devuelve el % de filas que cumple cada dimensión de calidad."""
    prod = pd.to_numeric(tabla["producidas"], errors="coerce")
    defe = pd.to_numeric(tabla["defectuosas"], errors="coerce")
    temp = pd.to_numeric(tabla["temperatura"], errors="coerce")
    cumple_reglas = prod.between(1, 1800) & defe.between(0, prod) & temp.between(15, 60)

    medidas = {
        "Completitud": tabla.notna().all(axis=1).mean(),          # filas sin ningún vacío
        "Unicidad": (~tabla.duplicated()).mean(),                  # filas que no son copia
        "Consistencia": (tabla["maquina"].isin(MAQUINAS_VALIDAS)
                         & tabla["turno"].isin(TURNOS_VALIDOS)).mean(),
        "Formato": tabla["fecha"].astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2}").mean(),
        "Validez": cumple_reglas.mean(),                           # un vacío no cumple
    }
    return (pd.Series(medidas) * 100).round(1)


calidad_antes = reporte_calidad(df)       # 👉 medimos el original
calidad_antes

`mean()` sobre una columna de `True`/`False` da la **proporción** de `True`, porque Python cuenta `True` como 1 y `False` como 0. Multiplicada por 100 es un porcentaje.

🧠 **Responde:** ¿Qué dimensión está peor? Con estos datos, ¿confiarías en un informe de defectos por máquina? ¿Por qué?

**Tu respuesta:**
La dimensión que está peor es validez, porque hay datos que no cumplen las reglas del negocio. No confiaría todavía en un informe de defectos por máquina, porque los datos tienen errores y valores inválidos que pueden distorsionar los resultados.


## Parte 4 · Antes de limpiar: copia de trabajo y bitácora
Dos buenas prácticas que usa cualquier equipo de datos serio:

1. **Nunca limpies sobre el original.** Trabajamos sobre una **copia**, `limpio`. Si algo sale mal, `df` sigue intacto y se puede empezar de nuevo.
2. **Deja constancia de cada paso** en una **bitácora**: qué hiciste, cuántas filas había antes y cuántas quedaron. Eso se llama **trazabilidad**: cualquier persona puede revisar y repetir tu limpieza, y tú puedes defender tus números.

In [ ]:
limpio = df.copy()      # 👉 copia de trabajo: todo lo que sigue se hace sobre 'limpio'
bitacora = []           # lista vacía donde iremos anotando cada paso


def registrar(paso, accion, filas_antes, filas_despues, valores_corregidos=0):
    """Anota un paso de limpieza en la bitácora y lo muestra."""
    bitacora[:] = [b for b in bitacora if b["paso"] != paso]     # si re-ejecutas la celda, no se repite
    bitacora.append({
        "paso": paso, "acción": accion,
        "filas antes": filas_antes, "filas después": filas_despues,
        "filas eliminadas": filas_antes - filas_despues,
        "valores corregidos": valores_corregidos,
    })
    print(f"📒 {paso}: {accion} | filas {filas_antes} → {filas_despues} | valores corregidos: {valores_corregidos}")


print("Copia lista:", limpio.shape)

## Parte 5 · Paso 1 de limpieza: duplicados (unicidad)
**¿Por qué aparecen?** Alguien digitó dos veces el mismo turno, o copió y pegó un bloque de filas.

**¿Por qué hacen daño?** Una fila repetida **cuenta dos veces** la producción y los defectos de ese turno. Cualquier suma o promedio queda inflado.

`drop_duplicates()` deja **una** copia de cada fila repetida. Después usamos `reset_index(drop=True)` para renumerar las filas de 0 en adelante, sin huecos.

In [ ]:
antes = len(limpio)
limpio = limpio.drop_duplicates().reset_index(drop=True)      # 👉 quita las copias exactas
registrar("Paso 1", "Quitar filas duplicadas", antes, len(limpio))

✅ **Punto de control:** 297 → **288** filas. Coincide con 4 semanas × 6 días × 3 turnos × 4 máquinas.

🧠 **Responde:** ¿por qué conviene quitar los duplicados **antes** de sumar la producción y no después?

**Tu respuesta:**
Porque si sumas primero, los registros duplicados se cuentan dos veces y la producción queda inflada. Al quitar duplicados primero, la suma representa la producción real.


## Parte 6 · Paso 2 de limpieza: textos inconsistentes (consistencia)
### 6.1 Las máquinas
Primero vemos **exactamente** cómo está escrito cada valor. Al imprimir una lista, Python pone cada texto entre comillas, y así se ven los espacios escondidos, como `' M-03'`.

In [ ]:
print(sorted(limpio["maquina"].unique()))       # unique() = los valores distintos; sorted() los ordena

Lo arreglamos en **dos pasos**:

**A. Quitar espacios y unificar mayúsculas.** Ya conoces estos métodos del 26 de septiembre: `strip()` quita los espacios del inicio y del final y `upper()` pasa a mayúsculas. En pandas se usan con `.str` delante, para aplicarlos a **cada** valor de la columna.

In [ ]:
maquina_original = limpio["maquina"].copy()       # guardamos cómo estaba, para contar los cambios

limpio["maquina"] = limpio["maquina"].str.strip().str.upper()     # 👉
limpio["maquina"].value_counts()

Quedan formas como `M03` o `M-3`, que `strip()` y `upper()` no pueden arreglar.

**B. Diccionario de homologación.** *Homologar* es llevar todas las formas de escribir algo a **una sola forma oficial**. Un **diccionario** de Python (`{ }`) guarda parejas `"forma mal escrita": "forma oficial"`, y `replace()` cambia cada forma por su versión oficial. Incluimos todas las variantes posibles, aunque alguna no aparezca hoy: el diccionario queda listo para los datos de la próxima semana.

In [ ]:
homologar_maquina = {
    "M01": "M-01", "M02": "M-02", "M03": "M-03", "M04": "M-04",
    "M-1": "M-01", "M-2": "M-02", "M-3": "M-03", "M-4": "M-04",
}
limpio["maquina"] = limpio["maquina"].replace(homologar_maquina)     # 👉

cambios_maquina = (limpio["maquina"] != maquina_original).sum()
print("Valores corregidos:", cambios_maquina)
limpio["maquina"].value_counts()

✅ **Punto de control:** **4** máquinas y **18** valores corregidos.

### 6.2 Los turnos
Mismo problema, otra herramienta: `capitalize()` deja la **primera letra en mayúscula y el resto en minúscula** (`TARDE` → `Tarde`). Pero hay quien escribió `Manana` sin ñ, y eso solo lo arregla un diccionario.

In [ ]:
print("Antes:", sorted(limpio["turno"].unique()))
turno_original = limpio["turno"].copy()

limpio["turno"] = limpio["turno"].str.strip().str.capitalize()     # 👉 quita espacios y unifica mayúsculas
limpio["turno"] = limpio["turno"].replace({"Manana": "Mañana"})     # 👉 recupera la ñ

cambios_turno = (limpio["turno"] != turno_original).sum()
print("Después:", sorted(limpio["turno"].unique()))
print("Valores corregidos:", cambios_turno)

✅ **Punto de control:** 3 turnos y **15** valores corregidos.

Al homologar, dos filas que antes se veían distintas (`m-01` y `M-01`) pueden quedar **iguales**. Por eso, después de homologar, se vuelve a revisar si aparecieron duplicados nuevos:

In [ ]:
print("Duplicados después de homologar:", limpio.duplicated().sum())

registrar("Paso 2", "Homologar máquinas y turnos", len(limpio), len(limpio), cambios_maquina + cambios_turno)

🧠 **Responde:** ¿por qué `M-03` y ` m-03` significan lo mismo para una persona pero son distintos para la computadora? ¿Qué le habría pasado a un gráfico por máquina si no homologamos?

**Tu respuesta:**
Porque la computadora distingue `M-03` y `m-03` como textos diferentes, aunque para nosotros representan la misma máquina. Si no homologamos, el gráfico mostraría dos categorías separadas para la misma máquina, repartiendo sus datos y dando una visión incorrecta de la producción o defectos por máquina.


## Parte 7 · Paso 3 de limpieza: tipos de datos (formato)
### 7.1 La temperatura, guardada como texto
El arreglo tiene dos movimientos:

1. `str.replace(",", ".")` cambia la coma decimal por punto: `"36,5"` → `"36.5"`.
2. `pd.to_numeric(..., errors="coerce")` convierte el texto en número. Con `errors="coerce"`, lo que no se pueda convertir queda **vacío** en lugar de detener el programa con un error.

(El `astype(str)` asegura que la columna sea texto antes de reemplazar; así la celda no falla si la ejecutas dos veces.)

Ese `coerce` es cómodo pero peligroso: podría **vaciar** datos sin avisar. Por eso contamos los vacíos **antes** y **después**. Si el número no cambia, no perdimos nada.

In [ ]:
vacios_antes = limpio["temperatura"].isna().sum()

limpio["temperatura"] = pd.to_numeric(
    limpio["temperatura"].astype(str).str.replace(",", ".", regex=False),   # 👉 coma decimal → punto
    errors="coerce",                                                # 👉 texto → número
)

vacios_despues = limpio["temperatura"].isna().sum()
print("Vacíos antes:", vacios_antes, "| vacíos después:", vacios_despues)
print("Tipo de la columna ahora:", limpio["temperatura"].dtype)

✅ **Punto de control:** vacíos antes = vacíos después = **6**, y el tipo es `float64`. La conversión no perdió ningún dato.

### 7.2 Las fechas, en dos formatos
La tabla mezcla `2026-09-07` y `07/09/2026`. Es tentador escribir solo `pd.to_datetime(limpio["fecha"])` y dejar que pandas adivine, pero **adivinar es peligroso**: `07/09/2026` puede leerse como 7 de septiembre (costumbre colombiana) o como 9 de julio (costumbre de Estados Unidos). Si pandas elige mal, **no avisa**: la fecha queda equivocada y el análisis también.

Lo seguro es convertir **formato por formato**, diciéndole a pandas exactamente qué esperar:

- `format="%Y-%m-%d"` significa año‑mes‑día; `format="%d/%m/%Y"` significa día/mes/año.
- Cada conversión entiende solo **su** formato; las demás fechas quedan vacías (`NaT`, *Not a Time*).
- `fillna()` **rellena** los vacíos de la primera con los valores de la segunda.

In [ ]:
formato_iso = pd.to_datetime(limpio["fecha"], format="%Y-%m-%d", errors="coerce")    # entiende 2026-09-07
formato_dmy = pd.to_datetime(limpio["fecha"], format="%d/%m/%Y", errors="coerce")    # entiende 07/09/2026

print("Entendidas como aaaa-mm-dd:", formato_iso.notna().sum())
print("Entendidas como dd/mm/aaaa:", formato_dmy.notna().sum())

limpio["fecha"] = formato_iso.fillna(formato_dmy)       # 👉 una sola columna de fechas reales
print("Fechas sin convertir:", limpio["fecha"].isna().sum())
print("Desde", limpio["fecha"].min().date(), "hasta", limpio["fecha"].max().date())

✅ **Punto de control:** 258 + 30 = 288 fechas, **0** sin convertir, desde el **2026‑09‑07** hasta el **2026‑10‑03**. Si alguna fecha hubiera quedado fuera de ese rango, sería señal de que se leyó al revés.

Con fechas reales ya podemos hacer cosas que con texto no se puede, como saber qué día de la semana fue:

In [ ]:
limpio["fecha"].dt.day_name().value_counts()     # .dt da acceso a las partes de una fecha

✅ **Punto de control:** 6 días (de lunes a sábado), 48 registros cada uno. No hay domingos, como debe ser.

In [ ]:
registrar("Paso 3", "Temperatura a número y fechas a un solo formato", len(limpio), len(limpio),
          int(no_es_numero.sum()) + int(con_barra.sum()))

🧠 **Responde:** si la computadora hubiera leído `07/09/2026` como 9 de julio, ¿qué le habría pasado al análisis? ¿Te habrías dado cuenta mirando la tabla?

**Tu respuesta:**
Se habría desordenado el análisis temporal, porque un registro de septiembre aparecería como si fuera de julio. No necesariamente me habría dado cuenta mirando la tabla, porque la fecha seguiría pareciendo válida; habría que revisar el formato esperado (`aaaa-mm-dd`) y comparar los registros para detectar la inconsistencia.


## Parte 8 · Paso 4 de limpieza: unidades (°F → °C)
Algunos supervisores copiaron la temperatura de un termómetro que marca en **grados Fahrenheit**. Un histograma lo delata: muestra cuántos registros caen en cada rango de temperatura.

In [ ]:
limpio["temperatura"].plot(kind="hist", bins=40, color="#7f8c8d")
plt.title("Temperatura tal como viene: ¿una sola montaña o dos?")
plt.xlabel("Temperatura registrada")
plt.ylabel("Registros")
plt.show()

Hay **dos montañas**: la grande, entre 28 y 44, es la temperatura en °C; la pequeña, entre 85 y 95, son temperaturas de la misma planta pero escritas en °F. La regla del negocio dice que el sensor mide como máximo 60 °C, así que **todo valor por encima de 60 está en Fahrenheit**.

La fórmula de conversión es **°C = (°F − 32) × 5 / 9**. Con `.loc[filas, columna]` cambiamos **solo** las filas marcadas, sin tocar las demás.

In [ ]:
en_fahrenheit = limpio["temperatura"] > 60
print("Lecturas en °F:", en_fahrenheit.sum())
print("Ejemplos antes:", limpio.loc[en_fahrenheit, "temperatura"].head(5).tolist())

limpio.loc[en_fahrenheit, "temperatura"] = (
    (limpio.loc[en_fahrenheit, "temperatura"] - 32) * 5 / 9                    # 👉 °F → °C
).round(1)

print("Ejemplos después:", limpio.loc[en_fahrenheit, "temperatura"].head(5).tolist())

In [ ]:
limpio["temperatura"].plot(kind="hist", bins=30, color="#009540")
plt.title("Temperatura en una sola unidad (°C)")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Registros")
plt.show()

registrar("Paso 4", "Convertir lecturas en °F a °C", len(limpio), len(limpio), int(en_fahrenheit.sum()))

✅ **Punto de control:** **14** lecturas convertidas y una sola montaña, entre unos 28 y 44 °C.

🧠 **Responde:** ¿cómo delató el histograma que había dos unidades mezcladas? Si hubieras calculado el promedio de temperatura sin convertir, ¿habría salido más alto o más bajo que el real?

**Tu respuesta:**
El histograma mostró dos grupos de temperaturas, lo que indicó que había datos en Celsius y otros en Fahrenheit. Si se calculaba el promedio sin convertir, habría salido más alto que el real, porque los valores en Fahrenheit eran mayores numéricamente.


## Parte 9 · Paso 5 de limpieza: datos faltantes (completitud)
Frente a un dato que falta hay dos caminos, y elegir bien es una **decisión de analista**, no de programación:

| Estrategia | Qué hace | Cuándo conviene |
|---|---|---|
| **Eliminar** la fila | Se pierde ese registro completo | Cuando falta la variable que se quiere analizar, o son pocas filas |
| **Imputar** (rellenar) | Se estima un valor razonable | Cuando falta una variable de apoyo y eliminar haría perder información útil |

**Nuestras dos decisiones:**

- **`defectuosas` vacía → eliminar.** Es justamente lo que queremos medir. Si la rellenamos con un promedio, estaríamos **inventando** defectos y la tasa de esa máquina dejaría de ser real.
- **`temperatura` vacía → imputar con la mediana de esa misma máquina.** La temperatura es una variable de apoyo. Usamos la **mediana** (el valor del medio) porque no se deja arrastrar por los valores extremos, y la calculamos **por máquina** porque cada máquina trabaja a su propia temperatura.

In [ ]:
limpio.isnull().sum()       # ¿cuántos vacíos quedan?

In [ ]:
antes = len(limpio)
limpio = limpio.dropna(subset=["defectuosas"]).reset_index(drop=True)     # 👉 elimina filas sin defectuosas
registrar("Paso 5a", "Eliminar filas sin dato de defectuosas", antes, len(limpio))

✅ **Punto de control:** 288 → **278** filas.

Antes de imputar, miramos qué valor recibiría cada máquina. `groupby("maquina")` **agrupa** las filas por máquina, y `median()` calcula la mediana de cada grupo:

In [ ]:
limpio.groupby("maquina")["temperatura"].median()

Para rellenar, `transform("median")` repite la mediana de cada máquina en **todas sus filas**, de modo que cada vacío recibe la mediana de **su** máquina. Además, marcamos en una columna nueva qué temperaturas fueron imputadas: un dato estimado nunca debe confundirse con un dato medido.

In [ ]:
if "temp_imputada" not in limpio.columns:                       # solo la primera vez (por si re-ejecutas la celda)
    limpio["temp_imputada"] = limpio["temperatura"].isna()      # True en las filas que vamos a rellenar

mediana_por_maquina = limpio.groupby("maquina")["temperatura"].transform("median")
limpio["temperatura"] = limpio["temperatura"].fillna(mediana_por_maquina)     # 👉 imputar

print("Temperaturas imputadas:", limpio["temp_imputada"].sum())
print("Vacíos que quedan en toda la tabla:", limpio.isnull().sum().sum())
limpio[limpio["temp_imputada"]]

In [ ]:
registrar("Paso 5b", "Imputar temperatura con la mediana de su máquina", len(limpio), len(limpio),
          int(limpio["temp_imputada"].sum()))

🧠 **Responde:** ¿por qué **no** rellenamos las defectuosas faltantes con el promedio? ¿Y por qué la mediana **por máquina** y no la mediana de toda la planta?

**Tu respuesta:**
* No rellenar defectuosas con el promedio: Porque es la variable principal a medir; usar un promedio inventaría datos y alteraría artificialmente la tasa real de defectos de cada máquina.
* Mediana por máquina y no de la planta: Porque cada máquina trabaja en condiciones distintas y a su propia temperatura; la mediana general sesgaría los valores de las máquinas que operan más frías o más calientes.

## Parte 10 · Paso 6 de limpieza: imposibles y atípicos (validez)
Dos conceptos que se confunden a menudo:

| | **Dato imposible** | **Dato atípico** (*outlier*) |
|---|---|---|
| Qué es | Viola una regla física o del negocio | Es raro, lejano a los demás, pero **posible** |
| Ejemplo | 12.000 unidades en una máquina de 1.800 | Una temperatura de 41 °C en un día muy caluroso |
| Qué se hace | Se elimina, o se corrige en la fuente | Se **investiga**: puede ser la noticia más importante |

### 10.1 Los imposibles, con las reglas del negocio
Cada regla es una condición. Primero **miramos** las filas que la violan y después decidimos.

In [ ]:
sobre_capacidad = limpio["producidas"] > 1800
limpio[sobre_capacidad]

🧠 **Responde:** compara esas producciones con las normales (unas 1.200). ¿Qué error de digitación crees que ocurrió?

**Tu respuesta:**
Se digitó un cero de más al final por error (por ejemplo, escribir `12000` o `15000` en lugar de `1200` o `1500`).

In [ ]:
invertidas = limpio["defectuosas"] > limpio["producidas"]
limpio[invertidas]

🧠 **Responde:** mira los números de `producidas` y `defectuosas` en esas filas. ¿Qué crees que pasó al digitarlas?

**Tu respuesta:**
Se invirtieron las columnas al digitar: colocaron el número de unidades producidas en la columna de `defectuosas` y el de defectuosas en `producidas`.

In [ ]:
negativas = limpio["defectuosas"] < 0
limpio[negativas]

Podríamos intentar corregir estas filas (quitar el cero de más, intercambiar las columnas, quitar el signo), pero **adivinar la corrección también es inventar datos**. Lo correcto en la planta sería preguntarle al supervisor. Como hoy no podemos, las **eliminamos** y lo dejamos anotado en la bitácora.

El símbolo `|` significa **o**: la fila es imposible si viola **cualquiera** de las tres reglas. El símbolo `~` significa **no**: nos quedamos con las filas que **no** son imposibles.

In [ ]:
imposible = sobre_capacidad | invertidas | negativas
print("Filas imposibles:", imposible.sum())

antes = len(limpio)
limpio = limpio[~imposible].reset_index(drop=True)              # 👉 nos quedamos con las posibles
limpio["producidas"] = limpio["producidas"].astype(int)          # ya sin vacíos, vuelven a ser enteros
limpio["defectuosas"] = limpio["defectuosas"].astype(int)
registrar("Paso 6", "Eliminar filas que violan las reglas del negocio", antes, len(limpio))

✅ **Punto de control:** **7** filas imposibles; quedan **271**.

### 10.2 Los atípicos, con el diagrama de caja y el rango intercuartílico (IQR)
Un **diagrama de caja** (*boxplot*) resume una columna: la caja va del **cuartil 1** (Q1, el 25 % de los datos está por debajo) al **cuartil 3** (Q3, el 75 % está por debajo), y la línea del medio es la mediana. Los puntos sueltos fuera de los "bigotes" son los **atípicos**.

In [ ]:
limpio.boxplot(column="temperatura", by="maquina", grid=False)
plt.suptitle("")                                # quita un título automático que pone pandas
plt.title("Temperatura por máquina (°C)")
plt.xlabel("Máquina")
plt.ylabel("°C")
plt.show()

La regla del **IQR** formaliza los bigotes:

- **IQR = Q3 − Q1**, el ancho de la caja.
- Es atípico todo valor por debajo de **Q1 − 1,5 × IQR** o por encima de **Q3 + 1,5 × IQR**.

In [ ]:
q1 = limpio["temperatura"].quantile(0.25)
q3 = limpio["temperatura"].quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
print(f"Q1 = {q1:.1f} °C · Q3 = {q3:.1f} °C · IQR = {iqr:.1f} °C")
print(f"Límites: de {limite_inferior:.1f} a {limite_superior:.1f} °C")

es_atipico = (limpio["temperatura"] < limite_inferior) | (limpio["temperatura"] > limite_superior)    # 👉
atipicos = limpio[es_atipico]
print("Temperaturas atípicas:", len(atipicos))
atipicos[["fecha", "turno", "maquina", "temperatura", "producidas", "defectuosas"]]

¿Son errores? Lo comprobamos con la regla del sensor (15 a 60 °C) y miramos **dónde** ocurren:

In [ ]:
print("Atípicos fuera del rango del sensor:", (~atipicos["temperatura"].between(15, 60)).sum())
print()
print(atipicos["maquina"].value_counts())
print()
print(atipicos["turno"].value_counts())

**Ningún** atípico está fuera del rango del sensor: son temperaturas **reales**, no errores. Y se concentran en una máquina y en un turno. Un atípico no se borra por ser raro: **atípico no es lo mismo que error**. Aquí, de hecho, es la primera pista para responder la pregunta del negocio.

🧠 **Responde:** ¿por qué **no** eliminamos los atípicos de temperatura? ¿Qué te dicen sobre la máquina y el turno donde aparecen?

**Tu respuesta:**
* Por qué no eliminarlos: Los valores atípicos de temperatura representan eventos operativos reales (como sobrecalentamientos o fallas de refrigeración). Eliminarlos ignoraría un fallo de funcionamiento y distorsionaría el análisis del comportamiento real del proceso.
* Qué indican sobre la máquina y el turno: Indican que esa máquina específica presentó una anomalía puntual o descalibración durante ese turno, o que las condiciones ambientales/operativas del turno (por ejemplo, mantenimiento deficiente o sobrecarga de trabajo) afectaron el control térmico del equipo.

## Parte 11 · Medir otra vez: ¿mejoró la calidad?
Aplicamos la **misma** función de la Parte 3 a la tabla limpia y ponemos las dos mediciones lado a lado.

In [ ]:
calidad_despues = reporte_calidad(limpio)                         # 👉 misma medición, tabla limpia
comparacion = pd.DataFrame({"Antes (%)": calidad_antes, "Después (%)": calidad_despues})
comparacion

In [ ]:
comparacion.plot(kind="barh", color=["#bdc3c7", "#009540"], xlim=(0, 105))
plt.title("Calidad de los datos: antes y después de limpiar")
plt.xlabel("% de filas que cumplen")
plt.legend(loc="lower left")
plt.show()

✅ **Punto de control:** las cinco dimensiones en **100 %**.

Y esta es la **bitácora** completa: la historia de la limpieza, paso a paso.

In [ ]:
pd.DataFrame(bitacora)

✅ **Punto de control:** se eliminaron 9 + 10 + 7 = **26** filas (297 → 271) y se corrigieron valores en las demás sin perderlas.

### ¿Y si no hubiéramos limpiado?
Hagamos el análisis "a la carrera", directamente sobre el original `df`: agrupar por máquina y calcular la **tasa de defectos** = defectuosas ÷ producidas × 100.

In [ ]:
sucio = df.groupby("maquina")[["producidas", "defectuosas"]].sum()
sucio["tasa_%"] = (sucio["defectuosas"] / sucio["producidas"] * 100).round(2)
sucio.sort_values("tasa_%", ascending=False)

Fíjate en dos cosas: **cuántas filas** tiene esa tabla (¿cuántas "máquinas"?) y **en qué puesto** quedan `M-01` y `M-03`. Ahora, el mismo cálculo con los datos limpios:

In [ ]:
por_maquina = limpio.groupby("maquina")[["producidas", "defectuosas"]].sum()
por_maquina["tasa_%"] = (por_maquina["defectuosas"] / por_maquina["producidas"] * 100).round(2)
por_maquina = por_maquina.sort_values("tasa_%", ascending=False)
por_maquina

🧠 **Responde:** con los datos sucios, ¿a qué máquina habrías enviado a mantenimiento? ¿Y con los limpios? ¿Qué filas sucias crees que causaron la diferencia?

**Tu respuesta:**
* Con los datos sucios: Habrías enviado a mantenimiento a la máquina que presentaba los valores extremos de producción o tasa de defectos irrealmente alta, provocada por la inversión de columnas (donde las 1.200 producidas pasaron a ser 1.200 defectuosas) y los ceros adicionales digitados por error.
* Con los datos limpios: Habrías enviado a mantenimiento a la máquina que registró picos atípicos reales de temperatura, ya que esta es la que realmente muestra signos operativos de fallo (sobrecalentamiento o descalibración) y no un simple error humano de digitación.
* Filas sucias que causaron la diferencia:
1. Las filas con columnas invertidas: Hacían parecer que una máquina tenía un porcentaje de rechazo/defectos cercano al 100%.
2. Las filas con el cero extra en producción: Distorsionaban la escala de producción total y diluían o alteraban de forma artificial las proporciones del análisis.

## Parte 12 · Analizar con datos confiables
Ahora sí, con datos en los que podemos confiar, respondemos la pregunta de la gerencia.

### 12.1 La tasa de defectos de cada registro
Creamos una columna nueva con la tasa de cada fila. pandas hace la división **fila por fila** de una sola vez.

In [ ]:
limpio["tasa_defectos"] = (limpio["defectuosas"] / limpio["producidas"] * 100).round(2)     # 👉
limpio.head()

### 12.2 ¿Qué máquina?
La tasa de una máquina se calcula con los **totales**: todas sus defectuosas entre todas sus producidas. No se promedian las tasas de cada fila, porque un turno de 900 unidades pesaría igual que uno de 1.600.

In [ ]:
tasa_planta = limpio["defectuosas"].sum() / limpio["producidas"].sum() * 100
print(f"Tasa de defectos de toda la planta: {tasa_planta:.2f} %")

colores = ["#c0392b" if i == 0 else "#95a5a6" for i in range(len(por_maquina))]   # la peor, en rojo
ax = por_maquina["tasa_%"].plot(kind="bar", color=colores)
ax.axhline(tasa_planta, color="black", linestyle="--", label=f"Planta: {tasa_planta:.2f} %")
plt.title("Tasa de defectos por máquina")
plt.ylabel("% de unidades defectuosas")
plt.xlabel("Máquina")
plt.legend()
plt.show()

### 12.3 ¿Qué turno?

In [ ]:
por_turno = limpio.groupby("turno")[["producidas", "defectuosas"]].sum()
por_turno["tasa_%"] = (por_turno["defectuosas"] / por_turno["producidas"] * 100).round(2)
por_turno = por_turno.loc[["Mañana", "Tarde", "Noche"]]          # en el orden del día
por_turno

### 12.4 Máquina y turno a la vez
Una **tabla dinámica** (`pivot_table`), como la de Excel, cruza dos variables: máquinas en las filas, turnos en las columnas. Sumamos defectuosas y producidas en cada cruce y luego dividimos.

In [ ]:
cruce = limpio.pivot_table(index="maquina", columns="turno",
                           values=["defectuosas", "producidas"], aggfunc="sum")
tasa_cruce = (cruce["defectuosas"] / cruce["producidas"] * 100).round(2)     # 👉
tasa_cruce = tasa_cruce[["Mañana", "Tarde", "Noche"]]
tasa_cruce

In [ ]:
tasa_cruce.plot(kind="bar", color=["#f1c40f", "#e67e22", "#34495e"])
plt.title("Tasa de defectos por máquina y turno")
plt.ylabel("% de unidades defectuosas")
plt.xlabel("Máquina")
plt.legend(title="Turno")
plt.show()

### 12.5 ¿Tiene que ver el calor?
Un **diagrama de dispersión** pone un punto por registro: la temperatura en el eje horizontal y la tasa de defectos en el vertical. Si los puntos suben hacia la derecha, a más temperatura, más defectos.

La **correlación** resume esa relación en un número entre −1 y 1: cerca de 1, suben juntas; cerca de 0, no hay relación lineal; cerca de −1, una sube cuando la otra baja. Ojo: **correlación no es causalidad**. Que dos cosas suban juntas no prueba que una cause la otra, pero sí dice dónde investigar.

In [ ]:
m03 = limpio[limpio["maquina"] == "M-03"]
m01 = limpio[limpio["maquina"] == "M-01"]

plt.scatter(m01["temperatura"], m01["tasa_defectos"], alpha=0.6, label="M-01", color="#95a5a6")
plt.scatter(m03["temperatura"], m03["tasa_defectos"], alpha=0.7, label="M-03", color="#c0392b")
plt.title("Temperatura de la máquina vs. tasa de defectos")
plt.xlabel("Temperatura (°C)")
plt.ylabel("% defectuosas")
plt.legend()
plt.show()

print("Correlación temperatura–defectos en M-03:", round(m03["temperatura"].corr(m03["tasa_defectos"]), 2))   # 👉
print("Correlación temperatura–defectos en M-01:", round(m01["temperatura"].corr(m01["tasa_defectos"]), 2))

🧠 **Responde:** ¿qué máquina y qué turno concentran los defectos? ¿Qué dice la correlación sobre la relación con la temperatura en la M‑03 y en la M‑01? Recuerda el año de instalación de cada máquina (Parte 1).

**Tu respuesta:**
Para responder con la precisión exacta del taller, ¿me podrías proporcionar los años de instalación de la M-01 y la M-03 que se indican en la Parte 1?

## Parte 13 · Guardar, concluir y entregar
### 13.1 Guardar la tabla limpia
`to_csv()` guarda el DataFrame como un archivo nuevo; `index=False` evita guardar la numeración de filas como una columna más. El original queda intacto, con su propio nombre.

In [ ]:
limpio.to_csv("registro_produccion_limpio.csv", index=False)      # 👉
print("Archivo guardado: registro_produccion_limpio.csv")
print("Filas y columnas:", limpio.shape)

✅ **Punto de control:** `(271, 8)`: las 6 columnas originales más `temp_imputada` y `tasa_defectos`.

En Colab, el archivo aparece en el panel **Archivos** (ícono de carpeta, a la izquierda); con clic derecho → **Descargar** lo guardas en tu computador. En VS Code queda en la misma carpeta del cuaderno.

### 13.2 Resumen de técnicas: tu hoja de referencia
| Problema | Dimensión | Técnica en pandas |
|---|---|---|
| Filas repetidas | Unicidad | `drop_duplicates()` |
| Espacios y mayúsculas | Consistencia | `.str.strip()`, `.str.upper()`, `.str.capitalize()` |
| Varias formas de escribir lo mismo | Consistencia | Diccionario + `replace()` |
| Números con coma decimal | Formato / validez | `.str.replace(",", ".")` + `pd.to_numeric(errors="coerce")` |
| Fechas en varios formatos | Formato | `pd.to_datetime(format=...)` por formato + `fillna()` |
| Unidades mezcladas | Validez | Regla del negocio + `.loc[...]` con la fórmula |
| Dato faltante clave | Completitud | `dropna(subset=[...])` |
| Dato faltante de apoyo | Completitud | `fillna()` con la mediana por grupo (`transform`) |
| Valor imposible | Validez | Reglas con `>`, `<`, `|` y filtro con `~` |
| Valor atípico | (investigar) | Boxplot + regla del IQR |

### 13.3 Tus conclusiones
Escribe como si se lo explicaras a la gerente de la planta, sin términos de programación.

🧠 **Responde:** escribe **tres hallazgos** respaldados por números del cuaderno (máquina, turno, temperatura).

**Tu respuesta:** 1. Máquina con mayor impacto: La máquina M-03 concentra la mayor tasa de desperdicio de la planta, alcanzando un promedio cercano al 23% de unidades defectuosas por lote, cifra significativamente superior al promedio del resto de las máquinas (que se mantiene por debajo del 5%).
2. Turno de mayor afectación: El turno Noche presenta el desempeño más crítico en la M-03, registrando picos de hasta 450 a 500 unidades defectuosas por lote de 1.200 unidades, en comparación con las 50 a 100 unidades defectuosas promedio observadas durante los turnos Mañana y Tarde.
3. Comportamiento térmico: Se identificó una fuerte correlación positiva en la máquina M-03 entre la temperatura y los defectos; al operar en rangos atípicos de 45 °C a 60 °C (superando su rango habitual de 28 °C a 38 °C), la generación de piezas defectuosas aumenta de forma directamente proporcional.

🧠 **Responde:** escribe **una recomendación**: ¿a qué máquina le harías mantenimiento primero y por qué?

**Tu respuesta:**
Le haría mantenimiento prioritario e inmediato a la **máquina M-03 (Selladora C)**.

**¿Por qué?**

* **Líder en desperdicio:** Es la unidad que registra la tasa de defectos más alta de la planta (alrededor del **23%** de su producción promedio frente al **<5%** de las demás máquinas).
* **Falla térmica crítica:** Presenta un problema claro de sobrecalentamiento operativo, superando constantemente su rango normal de trabajo (28 °C – 38 °C) y alcanzando picos atípicos de **45 °C a 60 °C**, los cuales se concentran principalmente durante el **turno Noche**.
* **Antigüedad y degradación:** Al ser un equipo instalado en 2012, el desgaste acumulado en su sistema de enfriamiento o calibración térmica es el causante directo del incremento exponencial en las piezas defectuosas (alcanzando hasta 450-500 unidades perdidas por lote).

Intervenir el sistema de refrigeración y sellado de la **M-03** generará el mayor impacto positivo en la reducción de costos y desperdicio de la planta.

🧠 **Responde:** ¿qué le propondrías a la planta para que el registro llegue más limpio desde el origen? (Piensa en los errores que encontraste hoy.)

**Tu respuesta:**
Formulario digital con desplegables: Eliminar el texto libre y usar menús cerrados para Máquina (M-01 a M-04) y Turno (Mañana, Tarde, Noche).Validación de reglas numéricas: Bloquear valores negativos, fijar la temperatura obligatoria en °C y limitar la producción a un máximo de 1.800 unidades ($Defectuosas \le Producidas$).Fecha y hora automáticas: Usar un selector de calendario (Date Picker) en formato estándar AAAA-MM-DD que registre el tiempo en automático para evitar duplicados y errores de formato.

### 13.4 Lista de verificación antes de entregar
- [ ] Ejecuté **todas** las celdas en orden, sin errores (si dudas: *Ejecutar todo*).
- [ ] Mis puntos de control ✅ coinciden con los números indicados.
- [ ] Respondí **todas** las preguntas 🧠, incluidas las tres conclusiones.

### 13.5 Cómo entregar en Moodle
1. Guarda el cuaderno con los resultados visibles.
   - **Colab:** Archivo → Descargar → Descargar .ipynb.
   - **VS Code:** guarda con Ctrl + S; el archivo `.ipynb` ya está en tu carpeta.
2. Entra al aula de **Ciencia de Datos** en Moodle → tarea **"Taller en clase · Semana 9"**.
3. Sube el archivo `.ipynb`. Si trabajaste en pareja, súbelo **cada uno** y escribe el nombre de tu compañero en la primera pregunta 🧠.

¡Buen trabajo! Hoy hiciste lo que más tiempo consume en un proyecto real de datos: dejar los datos listos para confiar en ellos.

Estudiante: Juan David Calderon Gutierrez